<a href="https://colab.research.google.com/github/gkkg19/Deep-Learning/blob/main/Optimizers_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import time

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Initialize parameters
def init_params(n_features):
    np.random.seed(42)
    W = np.random.randn(n_features, 1) * 0.1
    b = 0.0
    return W, b

# Compute gradients
def gradients(X, y, W, b):
    m = X.shape[0]
    y = y.reshape(-1, 1)
    y_hat = sigmoid(X @ W + b)
    dO = (y_hat - y) * y_hat * (1 - y_hat)
    dW = X.T @ dO / m
    db = np.sum(dO) / m
    return dW, db

# Training framework
def train(X_train, y_train, X_test, y_test, optimizer, epochs=1000, lr=0.01, batch_size=32):
    W, b = init_params(X_train.shape[1])
    velocity_W = np.zeros_like(W)
    velocity_b = 0.0
    beta = 0.9

    start = time.time()
    for epoch in range(epochs):

        if optimizer == "GD":
            dW, db = gradients(X_train, y_train, W, b)
            W -= lr * dW
            b -= lr * db

        elif optimizer == "SGD":
            for i in range(X_train.shape[0]):
                Xi = X_train[i:i+1]
                yi = y_train[i:i+1]
                dW, db = gradients(Xi, yi, W, b)
                W -= lr * dW
                b -= lr * db

        elif optimizer == "MiniBatch":
            idx = np.random.permutation(X_train.shape[0])
            for i in range(0, X_train.shape[0], batch_size):
                Xb = X_train[idx[i:i+batch_size]]
                yb = y_train[idx[i:i+batch_size]]
                dW, db = gradients(Xb, yb, W, b)
                W -= lr * dW
                b -= lr * db

        elif optimizer == "Momentum":
            dW, db = gradients(X_train, y_train, W, b)
            velocity_W = beta * velocity_W + (1 - beta) * dW
            velocity_b = beta * velocity_b + (1 - beta) * db
            W -= lr * velocity_W
            b -= lr * velocity_b

    elapsed = time.time() - start

    train_pred = np.round(sigmoid(X_train @ W + b))
    test_pred = np.round(sigmoid(X_test @ W + b))

    return {
        "train_acc": accuracy_score(y_train, train_pred),
        "test_acc": accuracy_score(y_test, test_pred),
        "time": elapsed
    }


optimizers = ["GD", "SGD", "MiniBatch", "Momentum"]

print(f"{'Optimizer':<12} {'Train Acc':<10} {'Test Acc':<10} {'Time(s)':<8}")
print("-" * 45)

for opt in optimizers:
    res = train(X_train, y_train, X_test, y_test, optimizer=opt, epochs=1000, lr=0.01)
    print(f"{opt:<12} {res['train_acc']:<10.4f} {res['test_acc']:<10.4f} {res['time']:<8.2f}")


Optimizer    Train Acc  Test Acc   Time(s) 
---------------------------------------------
GD           0.9626     0.9561     0.27    
SGD          0.9912     0.9561     18.16   
MiniBatch    0.9846     0.9737     0.47    
Momentum     0.9626     0.9561     0.03    
